# Quote-Then-Judge — Qwen2.5-14B-Instruct Re-run

This notebook repeats the full QTJ pipeline (candidate-conditioned retrieval →
training-pair construction → LoRA fine-tuning → held-out judgement) with the
**base model swapped from Qwen2.5-7B-Instruct to Qwen2.5-14B-Instruct**.

Everything else is kept identical to the validated Qwen-7B pipeline, with one
important fix baked in from the start: **fold membership is always read
directly from the `fold` field stored in `gold_full.json`** — there is no
hash-based fold reconstruction anywhere in this notebook, so there is no risk
of the document-leakage bug found in the earlier Qwen-7B run.

**Compute environment for this run:**
- GPU: NVIDIA A100 (Google Colab Pro)
- Compute budget: 107.32 units available
- Model: `Qwen/Qwen2.5-14B-Instruct` (4-bit QLoRA)

**Efficiency note:** candidate-conditioned retrieval (chunking + embedding
with `multilingual-e5-large`) does **not depend on the coding LLM at all** —
it was already computed once during the Qwen-7B run and cached to
`evidence_qtj_fold4.json`. This notebook automatically reuses that cached
file instead of recomputing it, saving both time and compute units. It will
only recompute retrieval if that cache is missing.

Outputs from this run are saved to a **separate folder**
(`outputs_qwen14b/`) so they never overwrite your Qwen-7B results — you'll be
able to compare both models side by side afterwards.


In [1]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================

!pip -q install \
    "transformers>=4.44" \
    "accelerate>=0.33" \
    "peft>=0.12" \
    "trl>=0.9" \
    "bitsandbytes>=0.43" \
    "sentence-transformers>=3.0" \
    "datasets>=2.20" \
    "scikit-learn" \
    "tqdm" \
    "pandas" \
    "matplotlib"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 60.1 MB/s eta 0:00:00


In [2]:
# ============================================================
# CELL 2 — IMPORTS, SEED, GPU CHECK
# ============================================================

import os
import gc
import gzip
import json
import math
import random
import re
import time
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch

SEED = 20260817

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU is required for this experiment.")

gpu = torch.cuda.get_device_properties(0)
print("GPU:", gpu.name)
print(f"VRAM: {gpu.total_memory / 1e9:.1f} GB")
print(f"Compute capability: sm_{gpu.major}{gpu.minor}")


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB
Compute capability: sm_80


In [3]:
# ============================================================
# CELL 3 — DRIVE MOUNT + PROJECT PATHS + EXPERIMENT CONFIG
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/ERIC_Thesis/Quote_Then_Judge")

if not BASE.exists():
    raise FileNotFoundError(f"Project folder not found: {BASE}")

print("Project folder:", BASE)


# ============================================================
# EXPERIMENT CONFIG — QWEN2.5-14B
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"

HELD_OUT_FOLD = 4

# Retrieval (unchanged — model-independent, reused from the 7B run if cached)
TOP_CHUNKS = 3
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
MAX_DOCUMENT_CHUNKS = 400

# Pair construction
HARD_NEGATIVES = 3
EASY_NEGATIVES = 1

# Model
MAX_SEQ = 1536

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Training — bumped batch size vs. the 7B run since A100 has more headroom
# than the L4 used previously. Effective batch size (16) kept the same.
EPOCHS = 3
LR = 1e-4
BATCH = 2
ACCUM = 8

# Main experiment (no ablation)
ABLATION_NO_DEFINITION = False

DIMENSIONS = ["eric", "focus", "level", "maturity", "mechanism"]
MULTI_LABEL_DIMS = {"eric", "focus"}
SINGLE_LABEL_DIMS = {"level", "maturity", "mechanism"}

# Shared data + cached retrieval evidence live in the ORIGINAL outputs folder
# (these are model-independent and were produced by the Qwen-7B run).
SHARED_OUT = BASE / "outputs"
SHARED_OUT.mkdir(exist_ok=True, parents=True)

# This model's own artifacts (LoRA adapter, training log, corrected metrics)
# go in a SEPARATE folder so nothing overwrites the Qwen-7B results.
OUT = BASE / "outputs_qwen14b"
OUT.mkdir(exist_ok=True, parents=True)

DRIVE_FOLD = OUT / f"fold_{HELD_OUT_FOLD}"
DRIVE_FOLD.mkdir(parents=True, exist_ok=True)

print("\nConfiguration:")
print("MODEL:", MODEL_NAME)
print("HELD_OUT_FOLD:", HELD_OUT_FOLD)
print("DIMENSIONS:", DIMENSIONS)
print("Shared data/outputs:", SHARED_OUT)
print("This model's outputs:", OUT)


Mounted at /content/drive
Project folder: /content/drive/MyDrive/ERIC_Thesis/Quote_Then_Judge

Configuration:
MODEL: Qwen/Qwen2.5-14B-Instruct
HELD_OUT_FOLD: 4
DIMENSIONS: ['eric', 'focus', 'level', 'maturity', 'mechanism']
Shared data/outputs: /content/drive/MyDrive/ERIC_Thesis/Quote_Then_Judge/outputs
This model's outputs: /content/drive/MyDrive/ERIC_Thesis/Quote_Then_Judge/outputs_qwen14b


In [4]:
# ============================================================
# CELL 4 — FILE PATHS + LOAD SHARED PROJECT DATA
# ============================================================

DOC_PATH = BASE / "corpus_documents.json.gz"
GOLD_PATH = BASE / "gold_full.json"
VERDICT_PATH = BASE / "verdict_inputs.json"
ERIC_PATH = BASE / "eric_strategies.json"
DIMDEF_PATH = BASE / "dimension_definitions.json"

# Cached, model-independent retrieval evidence from the Qwen-7B run.
EVIDENCE_PATH = SHARED_OUT / "evidence_qtj_fold4.json"

required = [DOC_PATH, GOLD_PATH, VERDICT_PATH, ERIC_PATH, DIMDEF_PATH]
missing = [str(p) for p in required if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Missing required project files:\n" + "\n".join(missing)
    )

with gzip.open(DOC_PATH, "rt", encoding="utf-8") as f:
    docs = json.load(f)

with open(GOLD_PATH, "r", encoding="utf-8") as f:
    GOLD_ROWS = json.load(f)

with open(VERDICT_PATH, "r", encoding="utf-8") as f:
    verdict_inputs = json.load(f)

with open(ERIC_PATH, "r", encoding="utf-8") as f:
    eric_strategies = json.load(f)

with open(DIMDEF_PATH, "r", encoding="utf-8") as f:
    dimension_definitions = json.load(f)

print("Loaded successfully.")
print("Documents:", len(docs))
print("Gold records:", len(GOLD_ROWS))
print("Cached evidence file exists:", EVIDENCE_PATH.exists())


Loaded successfully.
Documents: 221
Gold records: 228
Cached evidence file exists: True


In [5]:
# ============================================================
# CELL 5 — ERIC DEFINITIONS + CANDIDATE CODEBOOK
# ============================================================

if "definitions" not in verdict_inputs:
    raise ValueError("No ERIC definitions found in verdict_inputs.json.")

ERIC_DEFS = dict(verdict_inputs["definitions"])
print("ERIC definitions available:", len(ERIC_DEFS))

CANDIDATES = {"eric": dict(ERIC_DEFS)}

for dim in ["focus", "level", "maturity", "mechanism"]:
    cfg = dimension_definitions.get(dim)
    if cfg is None:
        raise KeyError(f"Missing dimension definition for: {dim}")
    values = cfg.get("values", {})
    if not isinstance(values, dict):
        raise TypeError(
            f"Expected {dim}.values to be a dictionary, got {type(values).__name__}"
        )
    CANDIDATES[dim] = dict(values)

print("\nCandidate counts:")
for dim in DIMENSIONS:
    print(f"  {dim:12s}: {len(CANDIDATES[dim])}")


ERIC definitions available: 73

Candidate counts:
  eric        : 73
  focus       : 7
  level       : 3
  maturity    : 3
  mechanism   : 3


In [6]:
# ============================================================
# CELL 6 — TRUE FOLD ASSIGNMENT (NO HASH FUNCTION, EVER)
# ============================================================
#
# Fold membership is read directly from the `fold` field stored in
# gold_full.json. Every document ID a gold row references (its primary
# doc_id AND any linked doc_ids) is mapped to that row's fold. If the
# same document is ever claimed by two different folds, this raises
# immediately rather than silently proceeding with leaked data.

doc_to_fold = {}

for row in GOLD_ROWS:
    fold = row["fold"]
    linked_ids = row.get("doc_ids") or [row["doc_id"]]
    for did in linked_ids:
        if did in doc_to_fold and doc_to_fold[did] != fold:
            raise ValueError(
                f"Document {did} appears in multiple folds "
                f"({doc_to_fold[did]} and {fold}) in gold_full.json itself."
            )
        doc_to_fold[did] = fold

TEST_DOCS = {d for d, f in doc_to_fold.items() if f == HELD_OUT_FOLD}
TRAIN_DOCS = {d for d, f in doc_to_fold.items() if f != HELD_OUT_FOLD}

print("TRUE fold-4 test documents:", len(TEST_DOCS))
print("TRUE training documents:", len(TRAIN_DOCS))
print("Overlap (must be zero):", len(TEST_DOCS & TRAIN_DOCS))

assert len(TEST_DOCS & TRAIN_DOCS) == 0, "Leakage detected — stop."

n_records_fold4 = sum(1 for r in GOLD_ROWS if r["fold"] == HELD_OUT_FOLD)
print("Gold records with fold == 4:", n_records_fold4)


# ============================================================
# GOLD LABEL EXTRACTION
# ============================================================

def gold_values(row, dim):

    if dim == "eric":
        return set(row.get("eric_codes") or [])

    if dim == "focus":
        raw = row.get("focus") or ""
        parts = re.split(r"[/,;&]|\band\b", str(raw), flags=re.IGNORECASE)
        return {p.strip() for p in parts if p.strip()}

    raw = row.get(dim)
    if raw is None or str(raw).strip() == "":
        return set()

    raw_str = str(raw).strip().lower()

    for candidate in CANDIDATES[dim]:
        base = candidate.lower().split(" (")[0]
        if base == raw_str or base in raw_str:
            return {candidate}

    return set()


# doc_id -> gold row lookup (first record referencing that document)
gold_by_doc = {}
for row in GOLD_ROWS:
    linked_ids = row.get("doc_ids") or [row["doc_id"]]
    for did in linked_ids:
        gold_by_doc.setdefault(did, row)

print("\ndoc_to_fold entries:", len(doc_to_fold))
print("gold_by_doc entries:", len(gold_by_doc))


TRUE fold-4 test documents: 40
TRUE training documents: 180
Overlap (must be zero): 0
Gold records with fold == 4: 41

doc_to_fold entries: 220
gold_by_doc entries: 220


In [7]:
# ============================================================
# CELL 7 — TEXT PROCESSING UTILITIES
# ============================================================

def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = str(text or "")
    if not text.strip():
        return []

    chunks = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break
        start = end - overlap

    return chunks


def split_sentences(text):
    text = str(text or "")
    parts = re.split(r"(?<=[.!?])\s+|\n+", text)
    parts = [x.strip() for x in parts if len(x.strip()) > 25]
    return parts or ([text[:300]] if text.strip() else [])


def document_text(doc):
    if isinstance(doc, dict):
        return str(doc.get("text", ""))
    return str(doc)


print("Text utilities ready.")


Text utilities ready.


In [8]:
# ============================================================
# CELL 8 — CANDIDATE + RETRIEVAL EVIDENCE
#
# If evidence_qtj_fold4.json already exists (produced by the Qwen-7B
# run), it is reused as-is — retrieval only depends on the embedding
# model (multilingual-e5-large) and the documents/candidates, never on
# the coding LLM, so there is nothing to gain from recomputing it here.
# It is only computed fresh if genuinely missing.
# ============================================================

if EVIDENCE_PATH.exists():

    print("Found cached retrieval evidence — reusing it (no recompute needed).")
    with open(EVIDENCE_PATH, "r", encoding="utf-8") as f:
        evidence = json.load(f)
    print("Documents with evidence:", len(evidence))

else:

    print("No cached evidence found — computing retrieval from scratch.")

    from sentence_transformers import SentenceTransformer

    enc = SentenceTransformer("intfloat/multilingual-e5-large", device="cuda")
    enc.max_seq_length = 512

    ALL_CANDIDATES = [(dim, value) for dim in DIMENSIONS for value in CANDIDATES[dim]]
    print("Total candidate values:", len(ALL_CANDIDATES))

    CVEC = {}
    for dim in DIMENSIONS:
        names = list(CANDIDATES[dim].keys())
        queries = [f"query: {name}. {CANDIDATES[dim][name]}" for name in names]
        vectors = enc.encode(
            queries, normalize_embeddings=True, batch_size=32, show_progress_bar=True
        )
        for name, vector in zip(names, vectors):
            CVEC[(dim, name)] = np.asarray(vector)

    evidence = {}
    t0 = time.time()

    for i, (did, doc) in enumerate(sorted(docs.items()), 1):

        text = document_text(doc)
        chunks = chunk_text(text)[:MAX_DOCUMENT_CHUNKS]
        if not chunks:
            continue

        passage_vectors = np.asarray(enc.encode(
            ["passage: " + c for c in chunks],
            normalize_embeddings=True, batch_size=64, show_progress_bar=False
        ))

        per_document = {}

        for dim, value in ALL_CANDIDATES:

            candidate_vector = CVEC[(dim, value)]
            similarities = passage_vectors @ candidate_vector
            top_idx = np.argsort(-similarities)[:TOP_CHUNKS]
            selected_chunks = [chunks[j] for j in sorted(top_idx)]
            passage = "\n---\n".join(selected_chunks)

            best_chunk_index = int(top_idx[0])
            sents = split_sentences(chunks[best_chunk_index])

            if sents:
                sentence_vectors = np.asarray(enc.encode(
                    ["passage: " + s for s in sents],
                    normalize_embeddings=True, batch_size=32, show_progress_bar=False
                ))
                sentence_scores = sentence_vectors @ candidate_vector
                best_sentence = sents[int(np.argmax(sentence_scores))]
            else:
                best_sentence = chunks[best_chunk_index][:300]

            per_document[f"{dim}|{value}"] = {
                "passage": passage[:2500],
                "evidence": best_sentence[:300],
                "similarity": float(np.max(similarities)),
            }

        evidence[did] = per_document

        if i % 20 == 0 or i == len(docs):
            elapsed = time.time() - t0
            remaining = len(docs) - i
            eta = elapsed / i * remaining if i else 0
            print(f"{i}/{len(docs)} | {elapsed/60:.1f} min elapsed | {eta/60:.1f} min remaining", flush=True)

    EVIDENCE_PATH.write_text(json.dumps(evidence, ensure_ascii=False), encoding="utf-8")
    print("\nEvidence saved:", EVIDENCE_PATH)
    print("Documents with evidence:", len(evidence))


Found cached retrieval evidence — reusing it (no recompute needed).
Documents with evidence: 221


In [9]:
# ============================================================
# CELL 9 — TRAINING PAIRS (positives + hard/easy negatives)
#
# Fold is assigned per pair using the TRUE doc_to_fold mapping from
# Cell 6 — never a hash function.
# ============================================================

random.seed(SEED)

pairs = []

# Precompute candidate embeddings' dot products for negative ranking.
# We need CVEC even if evidence was loaded from cache, so build it here
# if it doesn't already exist in this session.
if "CVEC" not in globals():
    from sentence_transformers import SentenceTransformer
    enc = SentenceTransformer("intfloat/multilingual-e5-large", device="cuda")
    enc.max_seq_length = 512
    CVEC = {}
    for dim in DIMENSIONS:
        names = list(CANDIDATES[dim].keys())
        queries = [f"query: {name}. {CANDIDATES[dim][name]}" for name in names]
        vectors = enc.encode(queries, normalize_embeddings=True, batch_size=32, show_progress_bar=True)
        for name, vector in zip(names, vectors):
            CVEC[(dim, name)] = np.asarray(vector)

for did, per_doc_evidence in evidence.items():

    row = gold_by_doc.get(did)
    if row is None:
        continue

    doc_fold = doc_to_fold.get(did)
    if doc_fold is None:
        continue

    for dim in DIMENSIONS:

        positive = gold_values(row, dim) & set(CANDIDATES[dim])
        if not positive:
            continue

        negative_pool = [v for v in CANDIDATES[dim] if v not in positive]

        ranked_negatives = sorted(
            negative_pool,
            key=lambda value: -max(
                float(np.dot(CVEC[(dim, value)], CVEC[(dim, true_value)]))
                for true_value in positive
            )
        )

        n_hard = min(HARD_NEGATIVES * len(positive), len(ranked_negatives))
        hard = ranked_negatives[:n_hard]
        remaining = ranked_negatives[n_hard:]
        n_easy = min(EASY_NEGATIVES * len(positive), len(remaining))
        easy = random.sample(remaining, n_easy)

        selected = list(positive) + hard + easy

        for value in selected:

            key = f"{dim}|{value}"
            if key not in per_doc_evidence:
                continue

            item = per_doc_evidence[key]
            is_positive = value in positive

            pairs.append({
                "doc": did,
                "fold": doc_fold,
                "dim": dim,
                "value": value,
                "definition": CANDIDATES[dim][value],
                "passage": item["passage"],
                "evidence": item["evidence"] if is_positive else "",
                "label": "YES" if is_positive else "NO",
                "negtype": "pos" if is_positive else ("hard" if value in hard else "easy"),
            })

print("Total pairs:", f"{len(pairs):,}")
print("\nBy dimension:")
for dim, n in Counter(p["dim"] for p in pairs).items():
    print(f"  {dim:12s}: {n:,}")
print("\nBy label:")
print(Counter(p["label"] for p in pairs))
print("\nBy pair type:")
print(Counter(p["negtype"] for p in pairs))


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Total pairs: 11,545

By dimension:
  eric        : 8,553
  focus       : 1,069
  level       : 657
  maturity    : 660
  mechanism   : 606

By label:
Counter({'NO': 8718, 'YES': 2827})

By pair type:
Counter({'hard': 7311, 'pos': 2827, 'easy': 1407})


In [10]:
# ============================================================
# CELL 10 — TRAIN / TEST SPLIT
# ============================================================

train_pairs = [p for p in pairs if p["fold"] != HELD_OUT_FOLD]
test_pairs_preview = [p for p in pairs if p["fold"] == HELD_OUT_FOLD]

train_docs = {p["doc"] for p in train_pairs}
test_docs_preview = {p["doc"] for p in test_pairs_preview}

overlap = train_docs & test_docs_preview

print("Training pairs:", f"{len(train_pairs):,}")
print("Training documents:", len(train_docs))
print("(Held-out fold pairs, preview only — the actual evaluation test set")
print(" is rebuilt independently and more carefully in the final section.)")
print("Document overlap:", len(overlap))

if overlap:
    raise ValueError("LEAKAGE: documents occur in both train and held-out fold.")

print("No document leakage.")


Training pairs: 9,508
Training documents: 180
(Held-out fold pairs, preview only — the actual evaluation test set
 is rebuilt independently and more carefully in the final section.)
Document overlap: 0
No document leakage.


In [11]:
# ============================================================
# CELL 11 — PROMPTS
# ============================================================

def build_evidence_prompt(p):

    definition_text = ""
    if not ABLATION_NO_DEFINITION:
        definition_text = f"\n\nDEFINITION:\n{p['definition']}"

    return f"""You are coding an implementation policy document.

CANDIDATE LABEL:
{p['value']}
{definition_text}

POLICY PASSAGE:
{p['passage']}

TASK:
Quote the shortest exact span from the policy passage that provides evidence
for whether the candidate label applies.

If there is no supporting evidence, output:
NONE

Output only the quotation or NONE.
"""


def build_judge_prompt(p, generated_evidence):

    definition_text = ""
    if not ABLATION_NO_DEFINITION:
        definition_text = f"\n\nDEFINITION:\n{p['definition']}"

    return f"""You are coding an implementation policy document.

CANDIDATE LABEL:
{p['value']}
{definition_text}

POLICY PASSAGE:
{p['passage']}

EVIDENCE QUOTE:
{generated_evidence}

TASK:
Does the quoted evidence support applying the candidate label to this
policy passage?

Answer exactly:
YES
or
NO

ANSWER:"""


def build_target(p):
    evidence_text = p["evidence"] if p["label"] == "YES" and p["evidence"] else "NONE"
    return f"EVIDENCE: {evidence_text}\nANSWER: {p['label']}"


example = next(p for p in train_pairs if p["label"] == "YES")

print("=== EVIDENCE PROMPT ===")
print(build_evidence_prompt(example))
print("\n=== TRAINING TARGET ===")
print(build_target(example))


=== EVIDENCE PROMPT ===
You are coding an implementation policy document.

CANDIDATE LABEL:
Facilitation


DEFINITION:
A process of interactive problem solving and support that occurs in a context of a recognized need for improvement and a supportive interpersonal relationship

POLICY PASSAGE:
e a memb er of PIC/S (The 
Pharmaceutical Inspection Convention /The Pharmaceutical Inspection Scheme).

6 | P a g e 
 
 
3.3 The manufacture, sale and d istribution of fake, adulterated, harmful, un -registered, 
counterfeit, misbranded and substandard drugs and medical devices must be forbade and 
exemplary punishment bestowed upon people responsible for such offences. 
 
3.4 The selection, quantity fixation, procurement, storage and distribution system of drugs 
must be strengthened so that drugs are accessible to the public throughout the country. 
Appropriate preservation methods, such as temperature and humidity control, must be 
ensured at all drug wholesale sho ps or pharmacies or drug st

In [12]:
# ============================================================
# CELL 12 — MODEL: Qwen2.5-14B-Instruct + 4-bit QLoRA
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset as HFDataset

tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print("Tokenizer loaded.")

yes_tokens = tok.encode("YES", add_special_tokens=False)
no_tokens = tok.encode("NO", add_special_tokens=False)

print("YES token IDs:", yes_tokens)
print("NO token IDs:", no_tokens)

if len(yes_tokens) != 1 or len(no_tokens) != 1:
    raise ValueError("YES or NO is not represented as a single token.")

YES_ID = yes_tokens[0]
NO_ID = no_tokens[0]

use_bf16 = torch.cuda.is_bf16_supported()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
)

print("BF16 supported:", use_bf16)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 25,165,824 || all params: 14,795,199,488 || trainable%: 0.1701


In [13]:
# ============================================================
# CELL 13 — BUILD HUGGING FACE TRAINING DATASET
# ============================================================

train_records = []

for p in train_pairs:
    train_records.append({
        "text": build_evidence_prompt(p) + "\n\n" + build_target(p)
    })

ds_train = HFDataset.from_list(train_records)

print("Training examples:", len(ds_train))
print("\nExample:")
print(ds_train[0]["text"][:2000])


Training examples: 9508

Example:
You are coding an implementation policy document.

CANDIDATE LABEL:
Facilitation


DEFINITION:
A process of interactive problem solving and support that occurs in a context of a recognized need for improvement and a supportive interpersonal relationship

POLICY PASSAGE:
e a memb er of PIC/S (The 
Pharmaceutical Inspection Convention /The Pharmaceutical Inspection Scheme).

6 | P a g e 
 
 
3.3 The manufacture, sale and d istribution of fake, adulterated, harmful, un -registered, 
counterfeit, misbranded and substandard drugs and medical devices must be forbade and 
exemplary punishment bestowed upon people responsible for such offences. 
 
3.4 The selection, quantity fixation, procurement, storage and distribution system of drugs 
must be strengthened so that drugs are accessible to the public throughout the country. 
Appropriate preservation methods, such as temperature and humidity control, must be 
ensured at all drug wholesale sho ps or pharmacies 

In [14]:
# ============================================================
# CELL 14 — TRAINING (LoRA fine-tune of Qwen2.5-14B-Instruct)
#
# CRASH/DISCONNECT SAFE:
# - Checkpoints save directly to Google Drive (CHECKPOINT_PATH lives
#   under /content/drive/MyDrive/...), not local Colab disk, so they
#   survive a disconnect, timeout, or your laptop going to sleep.
# - save_strategy="steps" with a small save_steps writes a checkpoint
#   frequently (not just once per epoch), so at most a few minutes of
#   progress is ever at risk.
# - If this cell is RE-RUN after an interruption, it automatically
#   detects the latest checkpoint already on Drive and resumes
#   training from there instead of starting over from scratch.
# ============================================================

from trl import SFTTrainer, SFTConfig

CHECKPOINT_PATH = DRIVE_FOLD / "checkpoints"
LORA_PATH = DRIVE_FOLD / "lora_main"

CHECKPOINT_PATH.mkdir(parents=True, exist_ok=True)
LORA_PATH.mkdir(parents=True, exist_ok=True)

print("All training outputs will be saved to:", DRIVE_FOLD)

# --------------------------------------------------------------
# CHECK FOR AN EXISTING CHECKPOINT TO RESUME FROM
# --------------------------------------------------------------

existing_checkpoints = sorted(
    CHECKPOINT_PATH.glob("checkpoint-*"),
    key=lambda p: int(p.name.split("-")[-1])
)

resume_from = str(existing_checkpoints[-1]) if existing_checkpoints else None

if resume_from:
    print(f"\nFound existing checkpoint — RESUMING training from:")
    print(f"  {resume_from}")
else:
    print("\nNo existing checkpoint found — starting training from scratch.")

sft_config = SFTConfig(
    output_dir=str(CHECKPOINT_PATH),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    logging_steps=10,

    # Save frequently, not just once per epoch, so a disconnect
    # never costs more than a few minutes of progress.
    save_strategy="steps",
    save_steps=25,
    save_total_limit=3,   # keep last 3 checkpoints on Drive as a safety margin

    bf16=use_bf16,
    fp16=not use_bf16,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to=[],
    seed=SEED,
    dataset_text_field="text",
    max_length=MAX_SEQ,
    packing=False,
)

trainer = SFTTrainer(model=model, args=sft_config, train_dataset=ds_train)

print("\nStarting training...")
t0 = time.time()

# resume_from_checkpoint=None just starts fresh; passing a path resumes
# optimizer state, scheduler state, and step count from that checkpoint.
train_result = trainer.train(resume_from_checkpoint=resume_from)

elapsed = time.time() - t0
print(f"\nTraining completed in {elapsed / 60:.1f} minutes.")

model.save_pretrained(LORA_PATH)
tok.save_pretrained(LORA_PATH)
print("\nFinal LoRA adapter saved to:", LORA_PATH)

result_dict = {
    "model": MODEL_NAME,
    "held_out_fold": HELD_OUT_FOLD,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "batch_size": BATCH,
    "gradient_accumulation": ACCUM,
    "max_seq_length": MAX_SEQ,
    "train_pairs": len(train_pairs),
    "train_documents": len(train_docs),
    "elapsed_minutes": elapsed / 60,
    "resumed_from_checkpoint": resume_from,
}
if hasattr(train_result, "metrics"):
    result_dict["metrics"] = train_result.metrics

(DRIVE_FOLD / "training_result.json").write_text(
    json.dumps(result_dict, indent=2, ensure_ascii=False), encoding="utf-8"
)

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print("LoRA adapter:", LORA_PATH)


All training outputs will be saved to: /content/drive/MyDrive/ERIC_Thesis/Quote_Then_Judge/outputs_qwen14b/fold_4

No existing checkpoint found — starting training from scratch.


Adding EOS to train dataset:   0%|          | 0/9508 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/9508 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/9508 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/9508 [00:00<?, ? examples/s]

...

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.



Starting training...


Step,Training Loss
10,2.204782
20,1.977139
30,1.790007
40,1.859318
50,1.745129
60,1.582063
70,1.659846
80,1.569087
90,1.626637
100,1.631230


Step,Training Loss
10,2.204782
20,1.977139
30,1.790007
40,1.859318
50,1.745129
60,1.582063
70,1.659846
80,1.569087
90,1.626637
100,1.631230



Training completed in 321.8 minutes.

Final LoRA adapter saved to: /content/drive/MyDrive/ERIC_Thesis/Quote_Then_Judge/outputs_qwen14b/fold_4/lora_main

TRAINING COMPLETE
LoRA adapter: /content/drive/MyDrive/ERIC_Thesis/Quote_Then_Judge/outputs_qwen14b/fold_4/lora_main


In [1]:
from pathlib import Path
CHECKPOINT_PATH = Path("/content/drive/MyDrive/ERIC_Thesis/Quote_Then_Judge/outputs_qwen14b/fold_4/checkpoints")
for p in sorted(CHECKPOINT_PATH.glob("checkpoint-*")):
    print(p)

In [2]:
# ============================================================
# CELL 15 — FREE GPU MEMORY BEFORE FINAL RE-EVALUATION
#
# The corrected re-evaluation section below reloads the base model +
# LoRA adapter fresh (so it can be run standalone even after a
# runtime restart). We free the training-time model handle first to
# avoid holding two copies in GPU memory at once.
# ============================================================

try:
    del trainer
except NameError:
    pass

try:
    del model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

print("GPU memory freed. Ready for final corrected re-evaluation.")


GPU memory freed. Ready for final corrected re-evaluation.


In [1]:
import gc
import torch

---
## FINAL STEP — CORRECTED RE-EVALUATION (true fold field, no leakage)

Everything below is the same validated re-evaluation procedure used for the
Qwen-7B run, pointed at the new Qwen-14B LoRA adapter. It rebuilds the fold-4
test set directly from `gold_full.json`'s `fold` field (never a hash
function), reuses the cached retrieval evidence, reloads the freshly trained
adapter, runs inference with a live progress bar, and saves corrected metrics
to their own `outputs_qwen14b/fold4_metrics_plots_CORRECTED` folder.


In [3]:
# ============================================================
# CELL 16 — RE-EVALUATION SETUP (standalone-safe: re-mounts drive,
#            re-defines paths in case this section is run after a
#            runtime restart)
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import json
import gzip
import re
import time
import gc
from pathlib import Path
from collections import defaultdict, Counter

import torch
import numpy as np
import pandas as pd

BASE = Path("/content/drive/MyDrive/ERIC_Thesis/Quote_Then_Judge")
SHARED_OUT = BASE / "outputs"
OUT = BASE / "outputs_qwen14b"

GOLD_PATH = BASE / "gold_full.json"
VERDICT_PATH = BASE / "verdict_inputs.json"
DIMDEF_PATH = BASE / "dimension_definitions.json"
EVIDENCE_PATH = SHARED_OUT / "evidence_qtj_fold4.json"
LORA_PATH = OUT / "fold_4" / "lora_main"

assert GOLD_PATH.exists(), f"Missing: {GOLD_PATH}"
assert VERDICT_PATH.exists(), f"Missing: {VERDICT_PATH}"
assert DIMDEF_PATH.exists(), f"Missing: {DIMDEF_PATH}"
assert EVIDENCE_PATH.exists(), f"Missing: {EVIDENCE_PATH}"
assert LORA_PATH.exists(), f"Missing LoRA adapter: {LORA_PATH}"

print("All required files found.")

DIMENSIONS = ["eric", "focus", "level", "maturity", "mechanism"]
MULTI_LABEL_DIMS = {"eric", "focus"}
SINGLE_LABEL_DIMS = {"level", "maturity", "mechanism"}

HELD_OUT_FOLD = 4
MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"
MAX_SEQ = 1536
BATCH_SIZE = 4


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All required files found.


In [4]:
# ============================================================
# CELL 17 — LOAD GOLD DATA + DEFINITIONS (CANDIDATE CODEBOOK)
# ============================================================

with open(GOLD_PATH, "r", encoding="utf-8") as f:
    GOLD_ROWS = json.load(f)

with open(VERDICT_PATH, "r", encoding="utf-8") as f:
    verdict_inputs = json.load(f)

with open(DIMDEF_PATH, "r", encoding="utf-8") as f:
    dimension_definitions = json.load(f)

with open(EVIDENCE_PATH, "r", encoding="utf-8") as f:
    evidence_data = json.load(f)

print("Gold records:", len(GOLD_ROWS))
print("Evidence documents:", len(evidence_data))

ERIC_DEFS = dict(verdict_inputs["definitions"])

CANDIDATES = {"eric": dict(ERIC_DEFS)}

for dim in ["focus", "level", "maturity", "mechanism"]:
    cfg = dimension_definitions.get(dim)
    if cfg is None:
        raise KeyError(f"Missing dimension definition for: {dim}")
    values = cfg.get("values", {})
    CANDIDATES[dim] = dict(values)

print("\nCandidate counts:")
for dim in DIMENSIONS:
    print(f"  {dim:12s}: {len(CANDIDATES[dim])}")


Gold records: 228
Evidence documents: 221

Candidate counts:
  eric        : 73
  focus       : 7
  level       : 3
  maturity    : 3
  mechanism   : 3


In [5]:
# ============================================================
# CELL 18 — THE FIX: REBUILD FOLD-4 TEST DOCUMENTS FROM THE
#            TRUE STORED `fold` FIELD (NOT A HASH FUNCTION)
# ============================================================

doc_to_fold = {}

for row in GOLD_ROWS:
    fold = row["fold"]
    linked_ids = row.get("doc_ids") or [row["doc_id"]]
    for did in linked_ids:
        if did in doc_to_fold and doc_to_fold[did] != fold:
            raise ValueError(
                f"Document {did} appears in multiple folds "
                f"({doc_to_fold[did]} and {fold}) — leakage in "
                f"the source gold_full.json itself."
            )
        doc_to_fold[did] = fold

TEST_DOCS = {did for did, fold in doc_to_fold.items() if fold == HELD_OUT_FOLD}
TRAIN_DOCS = {did for did, fold in doc_to_fold.items() if fold != HELD_OUT_FOLD}

print("TRUE fold-4 test documents:", len(TEST_DOCS))
print("TRUE training documents:", len(TRAIN_DOCS))
print("Overlap (must be zero):", len(TEST_DOCS & TRAIN_DOCS))

assert len(TEST_DOCS & TRAIN_DOCS) == 0, "Leakage detected — stop."

n_records_fold4 = sum(1 for r in GOLD_ROWS if r["fold"] == HELD_OUT_FOLD)
print("Gold records with fold == 4:", n_records_fold4)


TRUE fold-4 test documents: 40
TRUE training documents: 180
Overlap (must be zero): 0
Gold records with fold == 4: 41


In [6]:
# ============================================================
# CELL 19 — GOLD LABEL EXTRACTION
# ============================================================

def gold_values(row, dim):

    if dim == "eric":
        return set(row.get("eric_codes") or [])

    if dim == "focus":
        raw = row.get("focus") or ""
        parts = re.split(r"[/,;&]|\band\b", str(raw), flags=re.IGNORECASE)
        return {p.strip() for p in parts if p.strip()}

    raw = row.get(dim)
    if raw is None or str(raw).strip() == "":
        return set()

    raw_str = str(raw).strip().lower()

    for candidate in CANDIDATES[dim]:
        base = candidate.lower().split(" (")[0]
        if base == raw_str or base in raw_str:
            return {candidate}

    return set()


gold_by_doc = {}
for row in GOLD_ROWS:
    linked_ids = row.get("doc_ids") or [row["doc_id"]]
    for did in linked_ids:
        gold_by_doc.setdefault(did, row)


In [7]:
# ============================================================
# CELL 20 — REBUILD test_pairs OVER THE CORRECT TEST DOCUMENT SET
# ============================================================

test_pairs = []

for doc_id in sorted(TEST_DOCS):

    if doc_id not in evidence_data:
        print(f"WARNING: no evidence for {doc_id}, skipping.")
        continue

    row = gold_by_doc.get(doc_id)
    if row is None:
        print(f"WARNING: no gold row for {doc_id}, skipping.")
        continue

    for key, item in evidence_data[doc_id].items():

        if "|" not in key:
            continue

        dim, value = key.split("|", 1)

        if dim not in DIMENSIONS:
            continue

        positives = gold_values(row, dim)
        label = int(value in positives)

        test_pairs.append({
            "doc": doc_id,
            "dim": dim,
            "value": value,
            "definition": CANDIDATES[dim].get(value, ""),
            "passage": item.get("passage", ""),
            "evidence": item.get("evidence", ""),
            "label": label,
        })

print("Corrected test pairs:", len(test_pairs))
print("\nBy dimension:")
for dim in DIMENSIONS:
    subset = [p for p in test_pairs if p["dim"] == dim]
    positives = sum(p["label"] for p in subset)
    print(f"  {dim:12s} | pairs = {len(subset):5d} | positive = {positives:4d}")


Corrected test pairs: 3560

By dimension:
  eric         | pairs =  2920 | positive =  315
  focus        | pairs =   280 | positive =   41
  level        | pairs =   120 | positive =   40
  maturity     | pairs =   120 | positive =   40
  mechanism    | pairs =   120 | positive =   33


In [8]:
# ============================================================
# CELL 21 — LOAD MODEL + TRAINED LoRA ADAPTER (Qwen2.5-14B)
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

YES_ID = tok.encode("YES", add_special_tokens=False)
NO_ID = tok.encode("NO", add_special_tokens=False)
assert len(YES_ID) == 1 and len(NO_ID) == 1
YES_ID, NO_ID = YES_ID[0], NO_ID[0]

use_bf16 = torch.cuda.is_bf16_supported()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)

model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()

print("Model + LoRA adapter loaded from:", LORA_PATH)


model.safetensors.index.json:   0%|          | 0.00/47.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model + LoRA adapter loaded from: /content/drive/MyDrive/ERIC_Thesis/Quote_Then_Judge/outputs_qwen14b/fold_4/lora_main


In [11]:
!pip install -U bitsandbytes>=0.46.1

In [9]:
# ============================================================
# CELL 22 — INFERENCE (single-pass judge using retrieved evidence,
#            with a live progress bar)
# ============================================================

from tqdm.auto import tqdm

predictions = []

t0 = time.time()

n_batches = (len(test_pairs) + BATCH_SIZE - 1) // BATCH_SIZE

progress = tqdm(
    range(0, len(test_pairs), BATCH_SIZE),
    total=n_batches,
    desc="Judging fold-4 pairs",
)

for start in progress:

    batch = test_pairs[start:start + BATCH_SIZE]

    prompts = []
    for p in batch:
        prompt = f"""You are an expert implementation-science coder.

Candidate label:
{p['value']}

Definition:
{p['definition']}

Policy passage:
{p['passage']}

Evidence quote:
{p['evidence']}

Question:
Does the quoted evidence support assigning the candidate label to this passage?

Answer only YES or NO.
"""
        prompts.append(prompt)

    inputs = tok(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        logits = model(**inputs).logits
        last_pos = inputs["attention_mask"].sum(dim=1) - 1
        idx = torch.arange(logits.shape[0], device=logits.device)
        last_logits = logits[idx, last_pos, :]

        probs = torch.softmax(
            torch.stack([last_logits[:, NO_ID], last_logits[:, YES_ID]], dim=1),
            dim=1,
        )
        yes_probs = probs[:, 1].float().detach().cpu().numpy()

    for p, prob in zip(batch, yes_probs):
        predictions.append({
            "doc_id": p["doc"],
            "dim": p["dim"],
            "value": p["value"],
            "yes_prob": float(prob),
            "label": p["label"],
        })

    del inputs, logits, last_logits, probs
    if start % (BATCH_SIZE * 10) == 0:
        torch.cuda.empty_cache()

    progress.set_postfix({
        "done": len(predictions),
        "elapsed_min": f"{(time.time()-t0)/60:.1f}",
    })

print(f"\nInference complete in {(time.time()-t0)/60:.1f} min.")
print("Predictions:", len(predictions))


Judging fold-4 pairs:   0%|          | 0/890 [00:00<?, ?it/s]


Inference complete in 9.0 min.
Predictions: 3560


In [10]:
# ============================================================
# CELL 23 — METRICS (macro/micro F1, precision, recall)
# ============================================================

from sklearn.metrics import f1_score, precision_score, recall_score

def normalise_level(x):
    x = str(x).strip()
    if x.startswith("Macro"):
        return "Macro"
    if x.startswith("Meso"):
        return "Meso"
    if x.startswith("Micro"):
        return "Micro"
    return x

gold_lookup = {}
for doc_id in TEST_DOCS:
    row = gold_by_doc.get(doc_id)
    if row is None:
        continue
    for dim in DIMENSIONS:
        vals = gold_values(row, dim)
        if dim == "level":
            vals = {normalise_level(v) for v in vals}
        gold_lookup[(doc_id, dim)] = vals

doc_scores = defaultdict(dict)
for p in predictions:
    value = p["value"]
    if p["dim"] == "level":
        value = normalise_level(value)
    doc_scores[(p["doc_id"], p["dim"])][value] = p["yes_prob"]

results = []

for dim in DIMENSIONS:

    dim_keys = [(d, dd) for (d, dd) in doc_scores if dd == dim]

    y_true, y_pred = [], []

    if dim in MULTI_LABEL_DIMS:

        labels = sorted(set(v for k in dim_keys for v in doc_scores[k]))

        for doc, d in dim_keys:
            gold_set = gold_lookup.get((doc, dim), set())
            scores = doc_scores[(doc, dim)]
            for label in labels:
                y_true.append(int(label in gold_set))
                y_pred.append(int(scores.get(label, 0) >= 0.5))

        macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
        precision = precision_score(y_true, y_pred, average="micro", zero_division=0)
        recall = recall_score(y_true, y_pred, average="micro", zero_division=0)

    else:

        for doc, d in dim_keys:
            gold_set = gold_lookup.get((doc, dim), set())
            if not gold_set:
                continue
            gold_label = next(iter(gold_set))
            scores = doc_scores[(doc, dim)]
            predicted_label = max(scores, key=scores.get)
            y_true.append(gold_label)
            y_pred.append(predicted_label)

        labels = sorted(set(y_true) | set(y_pred))
        macro_f1 = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
        micro_f1 = f1_score(y_true, y_pred, labels=labels, average="micro", zero_division=0)
        precision = precision_score(y_true, y_pred, labels=labels, average="micro", zero_division=0)
        recall = recall_score(y_true, y_pred, labels=labels, average="micro", zero_division=0)

    results.append({
        "Dimension": dim,
        "Macro-F1": macro_f1,
        "Micro-F1": micro_f1,
        "Precision": precision,
        "Recall": recall,
        "N_docs": len(dim_keys),
    })

results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("QTJ FOLD-4 RESULTS — Qwen2.5-14B-Instruct (true fold split, no leakage)")
print("=" * 70)
print(results_df.to_string(index=False, formatters={
    "Macro-F1": "{:.4f}".format,
    "Micro-F1": "{:.4f}".format,
    "Precision": "{:.4f}".format,
    "Recall": "{:.4f}".format,
}))



QTJ FOLD-4 RESULTS — Qwen2.5-14B-Instruct (true fold split, no leakage)
Dimension Macro-F1 Micro-F1 Precision Recall  N_docs
     eric   0.2294   0.2305    0.2305 0.2305      40
    focus   0.3068   0.3107    0.3107 0.3107      40
    level   0.1852   0.3750    0.3750 0.3750      40
 maturity   0.3971   0.4000    0.4000 0.4000      40
mechanism   0.1026   0.1818    0.1818 0.1818      40


In [11]:
# ============================================================
# CELL 24 — SAVE CORRECTED RESULTS
# ============================================================

OUT_DIR = OUT / "fold4_metrics_plots_CORRECTED"
OUT_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(OUT_DIR / "fold4_all_dimension_metrics_CORRECTED.csv", index=False)

with open(OUT_DIR / "fold4_predictions_CORRECTED.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print("\nSaved corrected results to:", OUT_DIR)



Saved corrected results to: /content/drive/MyDrive/ERIC_Thesis/Quote_Then_Judge/outputs_qwen14b/fold4_metrics_plots_CORRECTED


In [12]:
import numpy as np

yes_probs = np.array([p["yes_prob"] for p in predictions])

print("yes_prob stats:")
print("  min:", yes_probs.min())
print("  max:", yes_probs.max())
print("  mean:", yes_probs.mean())
print("  median:", np.median(yes_probs))
print("  % below 0.1:", (yes_probs < 0.1).mean() * 100)
print("  % above 0.9:", (yes_probs > 0.9).mean() * 100)
print("  % near 0.5 (0.4-0.6):", ((yes_probs > 0.4) & (yes_probs < 0.6)).mean() * 100)

# Look at a few individual examples for a positive-label case
positives = [p for p in predictions if p["label"] == 1][:10]
for p in positives:
    print(f"{p['dim']:10s} | {p['value'][:40]:40s} | true=YES | yes_prob={p['yes_prob']:.3f}")

yes_prob stats:
  min: 2.3603439331054688e-05
  max: 1.0
  mean: 0.646885465269678
  median: 0.70703125
  % below 0.1: 4.578651685393258
  % above 0.9: 2.303370786516854
  % near 0.5 (0.4-0.6): 19.859550561797754
eric       | Tailor strategies                        | true=YES | yes_prob=0.652
eric       | Change liability laws                    | true=YES | yes_prob=0.320
eric       | Build a coalition                        | true=YES | yes_prob=0.797
eric       | Develop academic partnerships            | true=YES | yes_prob=0.594
eric       | Centralize technical assistance          | true=YES | yes_prob=0.621
eric       | Conduct ongoing training                 | true=YES | yes_prob=0.816
eric       | Assess for readiness and identify barrie | true=YES | yes_prob=0.754
eric       | Audit and provide feedback               | true=YES | yes_prob=0.531
focus      | AMR                                      | true=YES | yes_prob=0.562
level      | Macro                               

Re-running the evaluation code glithc in the previous one

In [13]:
import gc
import torch

try:
    del trainer
except NameError:
    pass

try:
    del model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

print("GPU memory freed. Ready for final corrected re-evaluation.")

GPU memory freed. Ready for final corrected re-evaluation.


In [14]:
from google.colab import drive
drive.mount("/content/drive")

import json
import gzip
import re
import time
import gc
from pathlib import Path
from collections import defaultdict, Counter

import torch
import numpy as np
import pandas as pd

BASE = Path("/content/drive/MyDrive/ERIC_Thesis/Quote_Then_Judge")
SHARED_OUT = BASE / "outputs"
OUT = BASE / "outputs_qwen14b"

GOLD_PATH = BASE / "gold_full.json"
VERDICT_PATH = BASE / "verdict_inputs.json"
DIMDEF_PATH = BASE / "dimension_definitions.json"
EVIDENCE_PATH = SHARED_OUT / "evidence_qtj_fold4.json"
LORA_PATH = OUT / "fold_4" / "lora_main"

assert GOLD_PATH.exists(), f"Missing: {GOLD_PATH}"
assert VERDICT_PATH.exists(), f"Missing: {VERDICT_PATH}"
assert DIMDEF_PATH.exists(), f"Missing: {DIMDEF_PATH}"
assert EVIDENCE_PATH.exists(), f"Missing: {EVIDENCE_PATH}"
assert LORA_PATH.exists(), f"Missing LoRA adapter: {LORA_PATH}"

print("All required files found.")

DIMENSIONS = ["eric", "focus", "level", "maturity", "mechanism"]
MULTI_LABEL_DIMS = {"eric", "focus"}
SINGLE_LABEL_DIMS = {"level", "maturity", "mechanism"}

HELD_OUT_FOLD = 4
MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"
MAX_SEQ = 1536
BATCH_SIZE = 4

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All required files found.


In [15]:
with open(GOLD_PATH, "r", encoding="utf-8") as f:
    GOLD_ROWS = json.load(f)

with open(VERDICT_PATH, "r", encoding="utf-8") as f:
    verdict_inputs = json.load(f)

with open(DIMDEF_PATH, "r", encoding="utf-8") as f:
    dimension_definitions = json.load(f)

with open(EVIDENCE_PATH, "r", encoding="utf-8") as f:
    evidence_data = json.load(f)

print("Gold records:", len(GOLD_ROWS))
print("Evidence documents:", len(evidence_data))

ERIC_DEFS = dict(verdict_inputs["definitions"])

CANDIDATES = {"eric": dict(ERIC_DEFS)}

for dim in ["focus", "level", "maturity", "mechanism"]:
    cfg = dimension_definitions.get(dim)
    if cfg is None:
        raise KeyError(f"Missing dimension definition for: {dim}")
    values = cfg.get("values", {})
    CANDIDATES[dim] = dict(values)

print("\nCandidate counts:")
for dim in DIMENSIONS:
    print(f"  {dim:12s}: {len(CANDIDATES[dim])}")

Gold records: 228
Evidence documents: 221

Candidate counts:
  eric        : 73
  focus       : 7
  level       : 3
  maturity    : 3
  mechanism   : 3


In [16]:
doc_to_fold = {}

for row in GOLD_ROWS:
    fold = row["fold"]
    linked_ids = row.get("doc_ids") or [row["doc_id"]]
    for did in linked_ids:
        if did in doc_to_fold and doc_to_fold[did] != fold:
            raise ValueError(
                f"Document {did} appears in multiple folds "
                f"({doc_to_fold[did]} and {fold}) — leakage in "
                f"the source gold_full.json itself."
            )
        doc_to_fold[did] = fold

TEST_DOCS = {did for did, fold in doc_to_fold.items() if fold == HELD_OUT_FOLD}
TRAIN_DOCS = {did for did, fold in doc_to_fold.items() if fold != HELD_OUT_FOLD}

print("TRUE fold-4 test documents:", len(TEST_DOCS))
print("TRUE training documents:", len(TRAIN_DOCS))
print("Overlap (must be zero):", len(TEST_DOCS & TRAIN_DOCS))

assert len(TEST_DOCS & TRAIN_DOCS) == 0, "Leakage detected — stop."

n_records_fold4 = sum(1 for r in GOLD_ROWS if r["fold"] == HELD_OUT_FOLD)
print("Gold records with fold == 4:", n_records_fold4)

TRUE fold-4 test documents: 40
TRUE training documents: 180
Overlap (must be zero): 0
Gold records with fold == 4: 41


In [17]:
def gold_values(row, dim):

    if dim == "eric":
        return set(row.get("eric_codes") or [])

    if dim == "focus":
        raw = row.get("focus") or ""
        parts = re.split(r"[/,;&]|\band\b", str(raw), flags=re.IGNORECASE)
        return {p.strip() for p in parts if p.strip()}

    raw = row.get(dim)
    if raw is None or str(raw).strip() == "":
        return set()

    raw_str = str(raw).strip().lower()

    for candidate in CANDIDATES[dim]:
        base = candidate.lower().split(" (")[0]
        if base == raw_str or base in raw_str:
            return {candidate}

    return set()


gold_by_doc = {}
for row in GOLD_ROWS:
    linked_ids = row.get("doc_ids") or [row["doc_id"]]
    for did in linked_ids:
        gold_by_doc.setdefault(did, row)

In [18]:
test_pairs = []

for doc_id in sorted(TEST_DOCS):

    if doc_id not in evidence_data:
        print(f"WARNING: no evidence for {doc_id}, skipping.")
        continue

    row = gold_by_doc.get(doc_id)
    if row is None:
        print(f"WARNING: no gold row for {doc_id}, skipping.")
        continue

    for key, item in evidence_data[doc_id].items():

        if "|" not in key:
            continue

        dim, value = key.split("|", 1)

        if dim not in DIMENSIONS:
            continue

        positives = gold_values(row, dim)
        label = int(value in positives)

        test_pairs.append({
            "doc": doc_id,
            "dim": dim,
            "value": value,
            "definition": CANDIDATES[dim].get(value, ""),
            "passage": item.get("passage", ""),
            "evidence": item.get("evidence", ""),
            "label": label,
        })

print("Corrected test pairs:", len(test_pairs))
print("\nBy dimension:")
for dim in DIMENSIONS:
    subset = [p for p in test_pairs if p["dim"] == dim]
    positives = sum(p["label"] for p in subset)
    print(f"  {dim:12s} | pairs = {len(subset):5d} | positive = {positives:4d}")

Corrected test pairs: 3560

By dimension:
  eric         | pairs =  2920 | positive =  315
  focus        | pairs =   280 | positive =   41
  level        | pairs =   120 | positive =   40
  maturity     | pairs =   120 | positive =   40
  mechanism    | pairs =   120 | positive =   33


In [25]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

YES_ID = tok.encode(" YES", add_special_tokens=False)
NO_ID = tok.encode(" NO", add_special_tokens=False)
assert len(YES_ID) == 1 and len(NO_ID) == 1
YES_ID, NO_ID = YES_ID[0], NO_ID[0]

print("YES_ID:", YES_ID, repr(tok.decode([YES_ID])))
print("NO_ID:", NO_ID, repr(tok.decode([NO_ID])))

use_bf16 = torch.cuda.is_bf16_supported()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)

model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()

print("Model + LoRA adapter loaded from:", LORA_PATH)

YES_ID: 14080 ' YES'
NO_ID: 5664 ' NO'


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Model + LoRA adapter loaded from: /content/drive/MyDrive/ERIC_Thesis/Quote_Then_Judge/outputs_qwen14b/fold_4/lora_main


In [32]:
from tqdm.auto import tqdm

# Confirm the correct next-token IDs for the evaluation prompt
YES_ID = tok.encode(" YES", add_special_tokens=False)[0]
NO_ID = tok.encode(" NO", add_special_tokens=False)[0]

print("YES_ID:", YES_ID, repr(tok.decode([YES_ID])))
print("NO_ID:", NO_ID, repr(tok.decode([NO_ID])))

predictions = []

t0 = time.time()

n_batches = (len(test_pairs) + BATCH_SIZE - 1) // BATCH_SIZE

progress = tqdm(
    range(0, len(test_pairs), BATCH_SIZE),
    total=n_batches,
    desc="Judging fold-4 pairs",
)

for start in progress:

    batch = test_pairs[start:start + BATCH_SIZE]

    prompts = []

    for p in batch:

        evidence_text = p["evidence"] if p["evidence"] else "NONE"

        # Same evaluation prompt as the corrected 7B evaluation
        prompt = f"""You are an expert implementation-science coder.

Candidate label:
{p['value']}

Definition:
{p['definition']}

Policy passage:
{p['passage']}

Evidence quote:
{evidence_text}

Question:
Does the quoted evidence support assigning the candidate label to this passage?

Answer only YES or NO.
"""

        prompts.append(prompt)

    inputs = tok(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ,
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.inference_mode():

        logits = model(**inputs).logits

        # Last non-padding token
        last_pos = inputs["attention_mask"].sum(dim=1) - 1

        idx = torch.arange(
            logits.shape[0],
            device=logits.device,
        )

        last_logits = logits[idx, last_pos, :]

        # Compare probability of NO vs YES
        probs = torch.softmax(
            torch.stack(
                [
                    last_logits[:, NO_ID],
                    last_logits[:, YES_ID],
                ],
                dim=1,
            ),
            dim=1,
        )

        yes_probs = (
            probs[:, 1]
            .float()
            .detach()
            .cpu()
            .numpy()
        )

    for p, prob in zip(batch, yes_probs):

        predictions.append(
            {
                "doc_id": p["doc"],
                "dim": p["dim"],
                "value": p["value"],
                "yes_prob": float(prob),
                "label": p["label"],
            }
        )

    del inputs, logits, last_logits, probs

    if start % (BATCH_SIZE * 10) == 0:
        torch.cuda.empty_cache()

    progress.set_postfix(
        {
            "done": len(predictions),
            "elapsed_min": f"{(time.time() - t0) / 60:.1f}",
        }
    )

print(
    f"\nInference complete in "
    f"{(time.time() - t0) / 60:.1f} min."
)

print("Predictions:", len(predictions))

YES_ID: 14080 ' YES'
NO_ID: 5664 ' NO'


Judging fold-4 pairs:   0%|          | 0/890 [00:00<?, ?it/s]


Inference complete in 9.0 min.
Predictions: 3560


In [34]:
from sklearn.metrics import f1_score, precision_score, recall_score

def normalise_level(x):
    x = str(x).strip()
    if x.startswith("Macro"):
        return "Macro"
    if x.startswith("Meso"):
        return "Meso"
    if x.startswith("Micro"):
        return "Micro"
    return x

gold_lookup = {}
for doc_id in TEST_DOCS:
    row = gold_by_doc.get(doc_id)
    if row is None:
        continue
    for dim in DIMENSIONS:
        vals = gold_values(row, dim)
        if dim == "level":
            vals = {normalise_level(v) for v in vals}
        gold_lookup[(doc_id, dim)] = vals

doc_scores = defaultdict(dict)
for p in predictions:
    value = p["value"]
    if p["dim"] == "level":
        value = normalise_level(value)
    doc_scores[(p["doc_id"], p["dim"])][value] = p["yes_prob"]

results = []

for dim in DIMENSIONS:

    dim_keys = [(d, dd) for (d, dd) in doc_scores if dd == dim]

    y_true, y_pred = [], []

    if dim in MULTI_LABEL_DIMS:

        labels = sorted(set(v for k in dim_keys for v in doc_scores[k]))

        for doc, d in dim_keys:
            gold_set = gold_lookup.get((doc, dim), set())
            scores = doc_scores[(doc, dim)]
            for label in labels:
                y_true.append(int(label in gold_set))
                y_pred.append(int(scores.get(label, 0) >= 0.5))

        macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
        precision = precision_score(y_true, y_pred, average="micro", zero_division=0)
        recall = recall_score(y_true, y_pred, average="micro", zero_division=0)

    else:

        for doc, d in dim_keys:
            gold_set = gold_lookup.get((doc, dim), set())
            if not gold_set:
                continue
            gold_label = next(iter(gold_set))
            scores = doc_scores[(doc, dim)]
            predicted_label = max(scores, key=scores.get)
            y_true.append(gold_label)
            y_pred.append(predicted_label)

        labels = sorted(set(y_true) | set(y_pred))
        macro_f1 = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
        micro_f1 = f1_score(y_true, y_pred, labels=labels, average="micro", zero_division=0)
        precision = precision_score(y_true, y_pred, labels=labels, average="micro", zero_division=0)
        recall = recall_score(y_true, y_pred, labels=labels, average="micro", zero_division=0)

    results.append({
        "Dimension": dim,
        "Macro-F1": macro_f1,
        "Micro-F1": micro_f1,
        "Precision": precision,
        "Recall": recall,
        "N_docs": len(dim_keys),
    })

results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("QTJ FOLD-4 RESULTS — Qwen2.5-14B-Instruct (true fold split, no leakage)")
print("=" * 70)
print(results_df.to_string(index=False, formatters={
    "Macro-F1": "{:.4f}".format,
    "Micro-F1": "{:.4f}".format,
    "Precision": "{:.4f}".format,
    "Recall": "{:.4f}".format,
}))


QTJ FOLD-4 RESULTS — Qwen2.5-14B-Instruct (true fold split, no leakage)
Dimension Macro-F1 Micro-F1 Precision Recall  N_docs
     eric   0.1959   0.1959    0.1959 0.1959      40
    focus   0.2500   0.2500    0.2500 0.2500      40
    level   0.1988   0.4250    0.4250 0.4250      40
 maturity   0.3839   0.3750    0.3750 0.3750      40
mechanism   0.1167   0.2121    0.2121 0.2121      40


In [35]:
import numpy as np
from collections import Counter

print("=" * 70)
print("14B PREDICTION DIAGNOSTIC")
print("=" * 70)

yes_probs = np.array([p["yes_prob"] for p in predictions])
gold = np.array([1 if p["label"] == "YES" else 0 for p in predictions])

print("\nYES probability:")
print("  min    :", yes_probs.min())
print("  max    :", yes_probs.max())
print("  mean   :", yes_probs.mean())
print("  median :", np.median(yes_probs))

print("\nGold labels:")
print(Counter(gold))

print("\nMean P(YES) by GOLD label:")
print("  Gold NO :", yes_probs[gold == 0].mean())
print("  Gold YES:", yes_probs[gold == 1].mean())

print("\nPredicted YES counts:")
for threshold in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    predicted_yes = (yes_probs >= threshold).sum()
    print(
        f"  threshold {threshold:.1f}: "
        f"{predicted_yes} / {len(predictions)}"
    )

14B PREDICTION DIAGNOSTIC

YES probability:
  min    : 9.775161743164062e-05
  max    : 1.0
  mean   : 0.6949114133802693
  median : 0.75390625

Gold labels:
Counter({np.int64(0): 3560})

Mean P(YES) by GOLD label:
  Gold NO : 0.6949114133802693
  Gold YES: nan

Predicted YES counts:
  threshold 0.2: 3390 / 3560
  threshold 0.3: 3331 / 3560
  threshold 0.4: 3257 / 3560
  threshold 0.5: 3146 / 3560
  threshold 0.6: 2829 / 3560
  threshold 0.7: 2305 / 3560
  threshold 0.8: 1142 / 3560


/tmp/ipykernel_3386/768966589.py:22: RuntimeWarning: Mean of empty slice.
  print("  Gold YES:", yes_probs[gold == 1].mean())
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [36]:
from collections import Counter
import numpy as np

print("=" * 70)
print("14B PREDICTION DIAGNOSTIC — CORRECTED")
print("=" * 70)

yes_probs = np.array([p["yes_prob"] for p in predictions])

print("\nRaw prediction label values:")
print(Counter(p["label"] for p in predictions))

print("\nYES probability:")
print("  min    :", yes_probs.min())
print("  max    :", yes_probs.max())
print("  mean   :", yes_probs.mean())
print("  median :", np.median(yes_probs))

# Handle numeric 0/1 labels correctly
gold = np.array([
    int(p["label"])
    for p in predictions
])

print("\nGold labels:")
print(Counter(gold))

print("\nMean P(YES) by GOLD label:")

if np.any(gold == 0):
    print("  Gold NO :", yes_probs[gold == 0].mean())

if np.any(gold == 1):
    print("  Gold YES:", yes_probs[gold == 1].mean())

print("\nPredicted YES counts:")

for threshold in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    predicted_yes = (yes_probs >= threshold).sum()

    print(
        f"  threshold {threshold:.1f}: "
        f"{predicted_yes} / {len(predictions)}"
    )

14B PREDICTION DIAGNOSTIC — CORRECTED

Raw prediction label values:
Counter({0: 3091, 1: 469})

YES probability:
  min    : 9.775161743164062e-05
  max    : 1.0
  mean   : 0.6949114133802693
  median : 0.75390625

Gold labels:
Counter({np.int64(0): 3091, np.int64(1): 469})

Mean P(YES) by GOLD label:
  Gold NO : 0.6940135762977662
  Gold YES: 0.7008287149197512

Predicted YES counts:
  threshold 0.2: 3390 / 3560
  threshold 0.3: 3331 / 3560
  threshold 0.4: 3257 / 3560
  threshold 0.5: 3146 / 3560
  threshold 0.6: 2829 / 3560
  threshold 0.7: 2305 / 3560
  threshold 0.8: 1142 / 3560


In [29]:
print("YES:", tok.encode("YES", add_special_tokens=False))
print(" YES:", tok.encode(" YES", add_special_tokens=False))

print("NO:", tok.encode("NO", add_special_tokens=False))
print(" NO:", tok.encode(" NO", add_special_tokens=False))

YES: [14004]
 YES: [14080]
NO: [8996]
 NO: [5664]


In [30]:
test_prompt = """You are an expert implementation-science coder.

Candidate label:
test

Definition:
test

Policy passage:
test

Evidence quote:
test

Question:
Does the quoted evidence support assigning the candidate label to this passage?

Answer only YES or NO.
"""

ids = tok.encode(test_prompt, add_special_tokens=False)

print("\nLast 15 tokens:")
for x in ids[-15:]:
    print(x, repr(tok.decode([x])))


Last 15 tokens:
1824 ' support'
60753 ' assigning'
279 ' the'
9144 ' candidate'
2383 ' label'
311 ' to'
419 ' this'
21085 ' passage'
1939 '?\n\n'
16141 'Answer'
1172 ' only'
14080 ' YES'
476 ' or'
5664 ' NO'
624 '.\n'


In [31]:
YES_ID = tok.encode(" YES", add_special_tokens=False)[0]
NO_ID = tok.encode(" NO", add_special_tokens=False)[0]

print("YES_ID:", YES_ID, repr(tok.decode([YES_ID])))
print("NO_ID:", NO_ID, repr(tok.decode([NO_ID])))

YES_ID: 14080 ' YES'
NO_ID: 5664 ' NO'


In [ ]:
OUT_DIR = OUT / "fold4_metrics_plots_CORRECTED"
OUT_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(OUT_DIR / "fold4_all_dimension_metrics_CORRECTED.csv", index=False)

with open(OUT_DIR / "fold4_predictions_CORRECTED.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print("\nSaved corrected results to:", OUT_DIR)

In [22]:
print("Tokenizer padding side:", tok.padding_side)

# Sanity check: decode what token is actually at "last_pos" for a few examples
sample_prompts = prompts[:2]  # reuse from your last inference batch, or rebuild 2 test prompts
inputs = tok(sample_prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ)
last_pos = inputs["attention_mask"].sum(dim=1) - 1

for i in range(len(sample_prompts)):
    token_id = inputs["input_ids"][i, last_pos[i]].item()
    print(f"Example {i}: token at computed last_pos = '{tok.decode([token_id])}'")

Tokenizer padding side: right
Example 0: token at computed last_pos = ':'
Example 1: token at computed last_pos = ':'


In [23]:
import numpy as np
from collections import Counter

ys = np.array([p["yes_prob"] for p in predictions])
gold = np.array([p["label"] for p in predictions])

print("YES probability:")
print(" min   :", ys.min())
print(" max   :", ys.max())
print(" mean  :", ys.mean())
print(" median:", np.median(ys))

print("\nGold labels:")
print(Counter(gold))

print("\nMean P(YES) by GOLD label:")
print("Gold NO :", ys[gold == 0].mean())
print("Gold YES:", ys[gold == 1].mean())

print("\nPredicted YES counts:")
for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    print(f"threshold {t:.1f}: {(ys >= t).sum()} / {len(ys)}")

YES probability:
 min   : 4.16184775531292e-09
 max   : 1.0
 mean  : 0.9272746312579717
 median: 1.0

Gold labels:
Counter({np.int64(0): 3091, np.int64(1): 469})

Mean P(YES) by GOLD label:
Gold NO : 0.9282515110012322
Gold YES: 0.9208363897091066

Predicted YES counts:
threshold 0.2: 3349 / 3560
threshold 0.3: 3325 / 3560
threshold 0.4: 3299 / 3560
threshold 0.5: 3285 / 3560
threshold 0.6: 3267 / 3560
threshold 0.7: 3256 / 3560
threshold 0.8: 3245 / 3560


In [24]:
ref = "EVIDENCE: test\nANSWER:"

ref_ids = tok.encode(ref, add_special_tokens=False)
yes_ids = tok.encode(ref + " YES", add_special_tokens=False)
no_ids = tok.encode(ref + " NO", add_special_tokens=False)

print("Prefix token count:", len(ref_ids))
print("With ' YES' appended, new tokens:", yes_ids[len(ref_ids):])
print("With ' NO' appended, new tokens:", no_ids[len(ref_ids):])

for t in yes_ids[len(ref_ids):]:
    print("YES-side token:", t, repr(tok.decode([t])))
for t in no_ids[len(ref_ids):]:
    print("NO-side token:", t, repr(tok.decode([t])))

print("\nCurrent YES_ID:", YES_ID, repr(tok.decode([YES_ID])))
print("Current NO_ID:", NO_ID, repr(tok.decode([NO_ID])))

Prefix token count: 9
With ' YES' appended, new tokens: [14080]
With ' NO' appended, new tokens: [5664]
YES-side token: 14080 ' YES'
NO-side token: 5664 ' NO'

Current YES_ID: 14004 'YES'
Current NO_ID: 8996 'NO'


In [37]:
# ============================================================
# CELL 1 — YES/NO TOKEN IDS
# ============================================================

YES_ID = tok.encode(" YES", add_special_tokens=False)
NO_ID  = tok.encode(" NO", add_special_tokens=False)

assert len(YES_ID) == 1, YES_ID
assert len(NO_ID) == 1, NO_ID

YES_ID = YES_ID[0]
NO_ID = NO_ID[0]

print("YES_ID:", YES_ID, repr(tok.decode([YES_ID])))
print("NO_ID :", NO_ID, repr(tok.decode([NO_ID])))

YES_ID: 14080 ' YES'
NO_ID : 5664 ' NO'


In [38]:
# ============================================================
# CELL 2 — EVIDENCE GENERATION
# ============================================================

def build_evidence_prompt_qtj(p):
    definition = p.get("definition", "")

    return f"""You are a policy analyst coding national health policy documents.

CANDIDATE DIMENSION:
{p["dim"]}

CANDIDATE CODE:
{p["value"]}

DEFINITION:
{definition}

POLICY PASSAGE:
{p["passage"]}

TASK:
Identify the shortest exact span from the policy passage that provides
evidence for whether the candidate code applies.

If there is no supporting evidence, output:
NONE

Output only the quotation or NONE.

EVIDENCE:"""

In [39]:
# ============================================================
# CELL 3 — TRUE TWO-STAGE QTJ INFERENCE
# ============================================================

import torch
import time
from tqdm.auto import tqdm

model.eval()

def build_judge_prompt_qtj(p, generated_evidence):
    definition = p.get("definition", "")

    return f"""You are a policy analyst coding national health policy documents.

CANDIDATE DIMENSION:
{p["dim"]}

CANDIDATE CODE:
{p["value"]}

DEFINITION:
{definition}

POLICY PASSAGE:
{p["passage"]}

QUOTED EVIDENCE:
{generated_evidence}

TASK:
Does the quoted evidence support assigning the candidate code to this
policy passage?

Answer YES only when the quoted evidence supports the candidate
according to its definition. Otherwise answer NO.

ANSWER:"""

In [40]:
# ============================================================
# RUN TWO-STAGE QTJ
# ============================================================

N_TEST = 20

# IMPORTANT:
# take 10 YES + 10 NO so the diagnostic is balanced
yes_pairs = [p for p in test_pairs if int(p["label"]) == 1]
no_pairs  = [p for p in test_pairs if int(p["label"]) == 0]

diagnostic_pairs = yes_pairs[:10] + no_pairs[:10]

print("Diagnostic pairs:", len(diagnostic_pairs))
print("Gold YES:", sum(int(p["label"]) == 1 for p in diagnostic_pairs))
print("Gold NO :", sum(int(p["label"]) == 0 for p in diagnostic_pairs))


# ------------------------------------------------------------
# STAGE 1: GENERATE EVIDENCE
# ------------------------------------------------------------

generated_results = []

for i, p in enumerate(tqdm(diagnostic_pairs, desc="Generating evidence")):

    prompt = build_evidence_prompt_qtj(p)

    inputs = tok(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tok.eos_token_id,
        )

    # Only decode newly generated tokens
    input_len = inputs["input_ids"].shape[1]

    generated_text = tok.decode(
        output_ids[0][input_len:],
        skip_special_tokens=True
    ).strip()

    # Clean possible formatting
    evidence = generated_text

    if "EVIDENCE:" in evidence:
        evidence = evidence.split("EVIDENCE:", 1)[1].strip()

    # Keep only first line if model adds extra text
    evidence = evidence.split("\n")[0].strip()

    if not evidence:
        evidence = "NONE"

    generated_results.append({
        "pair": p,
        "generated_evidence": evidence
    })

    del inputs, output_ids

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ------------------------------------------------------------
# STAGE 2: JUDGE GENERATED EVIDENCE
# ------------------------------------------------------------

diagnostic_predictions = []

for item in tqdm(generated_results, desc="Judging evidence"):

    p = item["pair"]
    evidence = item["generated_evidence"]

    judge_prompt = build_judge_prompt_qtj(
        p,
        evidence
    )

    inputs = tok(
        judge_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        logits = model(**inputs).logits

        # last actual input token
        last_pos = inputs["attention_mask"].sum(dim=1) - 1

        idx = torch.arange(
            logits.shape[0],
            device=logits.device
        )

        last_logits = logits[
            idx,
            last_pos,
            :
        ]

        two_logits = torch.stack(
            [
                last_logits[:, NO_ID],
                last_logits[:, YES_ID]
            ],
            dim=1
        )

        probs = torch.softmax(
            two_logits,
            dim=1
        )

        yes_prob = float(
            probs[0, 1]
            .detach()
            .cpu()
        )

    diagnostic_predictions.append({
        "doc": p["doc"],
        "dim": p["dim"],
        "value": p["value"],
        "gold": int(p["label"]),
        "yes_prob": yes_prob,
        "evidence": evidence,
        "passage": p["passage"],
    })

    del inputs, logits, last_logits, two_logits, probs

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("\nDONE:", len(diagnostic_predictions))

Diagnostic pairs: 20
Gold YES: 10
Gold NO : 10


Generating evidence:   0%|          | 0/20 [00:00<?, ?it/s]

Judging evidence:   0%|          | 0/20 [00:00<?, ?it/s]


DONE: 20


In [41]:
# ============================================================
# CELL 4 — INSPECT 20 QTJ DECISIONS
# ============================================================

for i, r in enumerate(diagnostic_predictions):

    print("\n" + "=" * 90)
    print(f"PAIR {i+1}")
    print("=" * 90)

    print("Dimension :", r["dim"])
    print("Candidate :", r["value"])
    print("GOLD      :", "YES" if r["gold"] == 1 else "NO")
    print("P(YES)    :", round(r["yes_prob"], 4))

    print("\nGENERATED EVIDENCE:")
    print(r["evidence"])

    print("\nPASSAGE:")
    print(r["passage"][:1000])


PAIR 1
Dimension : eric
Candidate : Tailor strategies
GOLD      : YES
P(YES)    : 0.0374

GENERATED EVIDENCE:
NONE

PASSAGE:
ectoral 
stakeholders (inter- and intra-sectoral) by- 
 
 Conducting regular meeting of National 
Steering Committee (NSC) (at list one per 
rear or earlier, if necessary) and National 
Technical Committee ( NTC) ( at least six 
monthly or earlier, if necessary) 
At least one NSC meeting and 2 
NTC meetings conducted per 
year 
DCU, CDC, DGHS 2017-2022 
(already in 
place and to 
be 
continued) 
 Sharing information and other relevant 
documents (formal/web -based open 
access) 
Web-based system for 
document sharing developed 
DCU, CDC, DGHS & other 
stakeholders 
Dec, 2017 
Dissemination meeting 
conducted/ Documents posted 
in the Web-based system 
DCU, CDC, DGHS & other 
stakeholders 
2017-2022 
 Carrying out advocacy program in joint 
venture 
Number of joint advocacy 
program conducted 
Lead- DCU, CDC, DGHS & All 
stakeholders 
2017-2022 
 Monitoring, 

In [42]:
# ============================================================
# CELL 5 — QTJ DIAGNOSTIC METRICS
# ============================================================

import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

probs = np.array([
    r["yes_prob"]
    for r in diagnostic_predictions
])

gold = np.array([
    r["gold"]
    for r in diagnostic_predictions
])

pred = (probs >= 0.5).astype(int)

print("=" * 70)
print("TWO-STAGE QTJ DIAGNOSTIC")
print("=" * 70)

print("\nGold:")
print("  YES:", int((gold == 1).sum()))
print("  NO :", int((gold == 0).sum()))

print("\nMean P(YES):")
print("  Gold YES:", probs[gold == 1].mean())
print("  Gold NO :", probs[gold == 0].mean())

print("\nPredictions @ 0.5:")
print("  YES:", int(pred.sum()))
print("  NO :", int((pred == 0).sum()))

print("\nMetrics:")
print("  F1       :", f1_score(gold, pred, zero_division=0))
print("  Precision:", precision_score(gold, pred, zero_division=0))
print("  Recall   :", recall_score(gold, pred, zero_division=0))

TWO-STAGE QTJ DIAGNOSTIC

Gold:
  YES: 10
  NO : 10

Mean P(YES):
  Gold YES: 0.05855712890625
  Gold NO : 0.04156494140625

Predictions @ 0.5:
  YES: 0
  NO : 20

Metrics:
  F1       : 0.0
  Precision: 0.0
  Recall   : 0.0


In [43]:
# ============================================================
# CELL — TRAINING-ALIGNED FULL QTJ GENERATION
# ============================================================

import torch
import re
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score

model.eval()

def build_full_qtj_prompt(p):
    # Keep passage reasonably short so the TASK + EVIDENCE + ANSWER
    # structure remains inside the model context.
    passage = str(p["passage"])[:1200]

    return f"""You are coding an implementation policy document.

CANDIDATE LABEL:
{p["value"]}

DEFINITION:
{p["definition"]}

POLICY PASSAGE:
{passage}

TASK:
Quote the shortest exact span from the policy passage that provides evidence
for whether the candidate label applies.

If there is no supporting evidence, output:
NONE

Output exactly in this format:

EVIDENCE: <quotation or NONE>
ANSWER: <YES or NO>

EVIDENCE:"""

In [44]:
# ============================================================
# GENERATE EVIDENCE + ANSWER IN ONE PASS
# ============================================================

generated_qtj = []

for p in tqdm(diagnostic_pairs, desc="Generating QTJ decisions"):

    prompt = build_full_qtj_prompt(p)

    inputs = tok(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    input_len = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )

    generated = tok.decode(
        output_ids[0][input_len:],
        skip_special_tokens=True
    ).strip()

    # --------------------------------------------------------
    # Extract EVIDENCE
    # --------------------------------------------------------

    evidence = "NONE"

    m = re.search(
        r"EVIDENCE:\s*(.*?)(?:\n|$)",
        generated,
        flags=re.IGNORECASE | re.DOTALL
    )

    if m:
        evidence = m.group(1).strip()

    # --------------------------------------------------------
    # Extract ANSWER
    # --------------------------------------------------------

    answer = None

    m = re.search(
        r"ANSWER:\s*(YES|NO)",
        generated,
        flags=re.IGNORECASE
    )

    if m:
        answer = m.group(1).upper()

    # fallback: search generated text
    if answer is None:
        if re.search(r"\bYES\b", generated, re.IGNORECASE):
            answer = "YES"
        elif re.search(r"\bNO\b", generated, re.IGNORECASE):
            answer = "NO"

    generated_qtj.append({
        "doc": p["doc"],
        "dim": p["dim"],
        "value": p["value"],
        "gold": int(p["label"]),
        "answer": answer,
        "evidence": evidence,
        "raw_generation": generated,
        "passage": p["passage"],
    })

    del inputs, output_ids

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nCompleted:", len(generated_qtj))

Generating QTJ decisions:   0%|          | 0/20 [00:00<?, ?it/s]


Completed: 20


In [45]:
# ============================================================
# INSPECT GENERATIONS
# ============================================================

for i, r in enumerate(generated_qtj):

    print("\n" + "=" * 100)
    print(f"PAIR {i+1}")
    print("=" * 100)

    print("DIMENSION :", r["dim"])
    print("CANDIDATE :", r["value"])
    print("GOLD      :", "YES" if r["gold"] else "NO")
    print("ANSWER    :", r["answer"])

    print("\nGENERATED:")
    print(r["raw_generation"])

    print("\nPASSAGE:")
    print(r["passage"][:1200])


PAIR 1
DIMENSION : eric
CANDIDATE : Tailor strategies
GOLD      : YES
ANSWER    : NO

GENERATED:
NONE
ANSWER: NO

ANSWER: NO

PASSAGE:
ectoral 
stakeholders (inter- and intra-sectoral) by- 
 
 Conducting regular meeting of National 
Steering Committee (NSC) (at list one per 
rear or earlier, if necessary) and National 
Technical Committee ( NTC) ( at least six 
monthly or earlier, if necessary) 
At least one NSC meeting and 2 
NTC meetings conducted per 
year 
DCU, CDC, DGHS 2017-2022 
(already in 
place and to 
be 
continued) 
 Sharing information and other relevant 
documents (formal/web -based open 
access) 
Web-based system for 
document sharing developed 
DCU, CDC, DGHS & other 
stakeholders 
Dec, 2017 
Dissemination meeting 
conducted/ Documents posted 
in the Web-based system 
DCU, CDC, DGHS & other 
stakeholders 
2017-2022 
 Carrying out advocacy program in joint 
venture 
Number of joint advocacy 
program conducted 
Lead- DCU, CDC, DGHS & All 
stakeholders 
2017-2022 
 Mo

In [46]:
# ============================================================
# DIAGNOSTIC METRICS
# ============================================================

valid = [
    r for r in generated_qtj
    if r["answer"] in {"YES", "NO"}
]

gold = np.array([
    r["gold"]
    for r in valid
])

pred = np.array([
    1 if r["answer"] == "YES" else 0
    for r in valid
])

print("\n" + "=" * 70)
print("TRAINING-ALIGNED QTJ DIAGNOSTIC")
print("=" * 70)

print("Total pairs :", len(generated_qtj))
print("Valid answers:", len(valid))

print("\nGold:")
print("  YES:", int((gold == 1).sum()))
print("  NO :", int((gold == 0).sum()))

print("\nPredicted:")
print("  YES:", int((pred == 1).sum()))
print("  NO :", int((pred == 0).sum()))

if len(valid):
    print("\nMetrics:")
    print("  F1       :", f1_score(gold, pred, zero_division=0))
    print("  Precision:", precision_score(gold, pred, zero_division=0))
    print("  Recall   :", recall_score(gold, pred, zero_division=0))


TRAINING-ALIGNED QTJ DIAGNOSTIC
Total pairs : 20
Valid answers: 20

Gold:
  YES: 10
  NO : 10

Predicted:
  YES: 0
  NO : 20

Metrics:
  F1       : 0.0
  Precision: 0.0
  Recall   : 0.0


In [47]:
# ============================================================
# EVIDENCE QUALITY CHECK
# ============================================================

def normalise_text(x):
    return re.sub(r"\s+", " ", str(x).lower()).strip()

quote_found = 0
quote_exact = 0

for r in generated_qtj:

    ev = r["evidence"]

    if ev and ev.upper() != "NONE":
        quote_found += 1

        if normalise_text(ev) in normalise_text(r["passage"]):
            quote_exact += 1

print("Non-NONE evidence:", quote_found, "/", len(generated_qtj))
print("Exact evidence in passage:", quote_exact, "/", len(generated_qtj))

if quote_found:
    print(
        "Evidence exact-match rate:",
        round(quote_exact / quote_found, 3)
    )

Non-NONE evidence: 0 / 20
Exact evidence in passage: 0 / 20


In [48]:
# ============================================================
# DIAGNOSTIC — IS RETRIEVAL ACTUALLY FINDING THE GOLD EVIDENCE?
# ============================================================

for r in generated_qtj:

    print("\n" + "=" * 100)
    print("CANDIDATE:", r["value"])
    print("GOLD:", "YES" if r["gold"] else "NO")
    print("-" * 100)

    print("RETRIEVED PASSAGE:")
    print(r["passage"])

    print("\nMODEL EVIDENCE:")
    print(r["evidence"])

    print("\nMODEL ANSWER:")
    print(r["answer"])


CANDIDATE: Tailor strategies
GOLD: YES
----------------------------------------------------------------------------------------------------
RETRIEVED PASSAGE:
ectoral 
stakeholders (inter- and intra-sectoral) by- 
 
 Conducting regular meeting of National 
Steering Committee (NSC) (at list one per 
rear or earlier, if necessary) and National 
Technical Committee ( NTC) ( at least six 
monthly or earlier, if necessary) 
At least one NSC meeting and 2 
NTC meetings conducted per 
year 
DCU, CDC, DGHS 2017-2022 
(already in 
place and to 
be 
continued) 
 Sharing information and other relevant 
documents (formal/web -based open 
access) 
Web-based system for 
document sharing developed 
DCU, CDC, DGHS & other 
stakeholders 
Dec, 2017 
Dissemination meeting 
conducted/ Documents posted 
in the Web-based system 
DCU, CDC, DGHS & other 
stakeholders 
2017-2022 
 Carrying out advocacy program in joint 
venture 
Number of joint advocacy 
program conducted 
Lead- DCU, CDC, DGHS & All 
stake